#### Transform Orders Data - String to JSON
1. Pre-Process the Json String to fix the Data Quality issues
2. Transform JSON String to JSON Object
3. Write transformed data to silver schema

In [0]:
select * from gizmobox_catalog_subbu.bronze.v_orders

##### 1. Pre-Process the Json String to fix the Data Quality issues

In [0]:
CREATE OR REPLACE TEMPORARY VIEW tv_orders_fixed as
select value,
    regexp_replace(value,'"order_date": (\\d{4}-\\d{2}-\\d{2})','"order_date": "\$1"') as fixed_value
from gizmobox_catalog_subbu.bronze.v_orders

In [0]:
select * from tv_orders_fixed

##### 2. Transform JSON String to JSON Object

In [0]:
select schema_of_json(fixed_value),
    fixed_value
from tv_orders_fixed
limit 1

In [0]:
select from_json(fixed_value, 'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>' ) as json_value,
    fixed_value
from tv_orders_fixed


##### 3. Write transformed data to silver schema

In [0]:

create table if not exists gizmobox_catalog_subbu.silver.orders_json
select from_json(fixed_value, 'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>' ) as json_value
from tv_orders_fixed



In [0]:
select * from gizmobox_catalog_subbu.silver.orders_json